# 13 · Final training on the jittered crops — session A
**Runtime: T4 GPU.** Run all. Two copies of this notebook (sessions A and B) can run at the same time; they share the work
through small lock files, so every run is trained exactly once. Resumable: after a disconnect just **Run all** again.

Data `data/processed/cbis_mass_256_jit` (notebook 11), same folds, model library (`qseg2`) and base settings
(batch 16, Adam 1e-4, up to 200 epochs, early stopping on val_loss after epoch 15, patience 10, best checkpoint on val_loss).

* **Primary set** (`results/p2_jit_runs_<PRIMARY>`): all 8 models with seed 0, plus the six G/P models with seeds 1 and 2 = 100 runs.
* **Sensitivity set** (`results/p2_jit_runs_<OTHER>`): the six G/P models with seeds 0, 1, 2 under the other learning-rate
  schedule = 90 runs. Starts automatically when the primary set is finished (set `RUN_SENSITIVITY = False` to skip).

`PRIMARY` (cell 1) chooses which learning-rate schedule is the primary one: `'train'` = ReduceLROnPlateau watches the training
loss, `'val'` = it watches the validation loss (as in the first submission). Everything else is identical. Both sessions must
use the same value.


In [ ]:
# 1 · Settings, Drive, paths
SESSION         = 'A'      # 'A' or 'B' (the other copy of this notebook uses the other letter)
PRIMARY         = 'train'  # learning-rate schedule of the primary set: 'train' or 'val'  (same in both sessions!)
RUN_SENSITIVITY = True     # after the primary set, train the 90 sensitivity runs under the other schedule

from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob, importlib, json, time
assert PRIMARY in ('train', 'val') and SESSION in ('A', 'B')
OTHER    = 'val' if PRIMARY == 'train' else 'train'
PROJECT  = '/content/drive/MyDrive/quantum-placement'
DATA_DIR = f'{PROJECT}/data/processed/cbis_mass_256_jit'
CODE_DIR = f'{PROJECT}/code'
ROOT     = {s: f'{PROJECT}/results/p2_jit_runs_{s}' for s in ('train', 'val')}
QSEG_DIR = '/content/drive/MyDrive/quantum-placement/code'            # folder that contains the qseg/ package
if not os.path.exists(f'{QSEG_DIR}/qseg/quantum_sim.py'):
    hits = glob.glob('/content/drive/MyDrive/**/qseg/quantum_sim.py', recursive=True)
    assert hits, 'qseg package not found on Drive'
    QSEG_DIR = os.path.dirname(os.path.dirname(hits[0]))
for d in ROOT.values(): os.makedirs(d, exist_ok=True)
assert os.path.exists(f'{DATA_DIR}/folds.csv'), 'Run notebook 11 first'
for p in (QSEG_DIR, CODE_DIR):
    if p not in sys.path: sys.path.insert(0, p)
cfgfile = f'{PROJECT}/results/p2_jit_primary.json'
if os.path.exists(cfgfile):
    assert json.load(open(cfgfile))['primary'] == PRIMARY, 'PRIMARY differs from the value used before - stop and check'
else:
    json.dump({'primary': PRIMARY}, open(cfgfile, 'w'))
print('session', SESSION, '| primary schedule:', PRIMARY, '->', ROOT[PRIMARY], '| sensitivity:', OTHER, '->', ROOT[OTHER])


In [ ]:
# 2 · Use the model library already on Drive (written by notebook 02; not rewritten here)
import importlib, qseg2; importlib.reload(qseg2)
import tensorflow as tf, numpy as np, pandas as pd
print('qseg2 from:', qseg2.__file__)
print('TF', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))
assert tf.config.list_physical_devices('GPU'), 'No GPU: Runtime -> Change runtime type -> T4 GPU'


In [ ]:
# 3 · Data + shared config (base settings of the qseg package)
from qseg.config import Config
cfg = Config(work_dir=ROOT[PRIMARY], max_epochs=200)   # v3: up to 200 epochs; batch 16, Adam 1e-4, early stop patience 10 after epoch 15
X, Y, DF = qseg2.load_cbis(DATA_DIR)
print(X.shape, Y.shape, 'ROIs:', len(DF), '| patients:', DF.patient.nunique())
print(DF.groupby('fold').agg(rois=('roi_id','size'), patients=('patient','nunique')).T.to_string())


In [ ]:
# 3c · Disk-space fix: Keras stores the Adam optimizer state inside best.weights.h5 (~95 MB per run).
# After each run, the weights are re-saved WITHOUT the optimizer state (~31 MB) and the new file is
# verified weight-by-weight against the original before it replaces it. Model weights are unchanged.
def shrink_weights(run, root):
    d = f'{root}/{run}'; big = f'{d}/best.weights.h5'; small = f'{d}/best_small.weights.h5'
    if not os.path.exists(big) or os.path.exists(f'{d}/.shrunk'):
        return
    if os.path.getsize(big) < 60e6:
        open(f'{d}/.shrunk', 'w').close(); return
    name = run.split('_fold')[0]
    tf.keras.backend.clear_session()
    m1 = qseg2.build_p2_model(name, (cfg.image_size, cfg.image_size, cfg.in_channels), cfg.n_qubits, cfg.n_q_layers)
    m1.load_weights(big)
    m1.save_weights(small)
    m2 = qseg2.build_p2_model(name, (cfg.image_size, cfg.image_size, cfg.in_channels), cfg.n_qubits, cfg.n_q_layers)
    m2.load_weights(small)
    w1, w2 = m1.get_weights(), m2.get_weights()
    assert len(w1) == len(w2) and all(np.array_equal(a, b) for a, b in zip(w1, w2)), f'{run}: weight check failed, original kept'
    os.replace(small, big)
    open(f'{d}/.shrunk', 'w').close()
    print(f'   {run}: weights re-saved without optimizer state ({os.path.getsize(big) / 1e6:.0f} MB)', flush=True)


In [ ]:
# 4 · Learning-rate schedule switch. qseg2.run_fold creates ReduceLROnPlateau(monitor='val_loss', ...);
# with MONITOR['v'] = 'loss' the same callback watches the training loss instead. Factor 0.5, patience 4, min_lr 1e-7,
# early stopping and the best checkpoint (both on val_loss) are unchanged.
_Base = tf.keras.callbacks.ReduceLROnPlateau
while getattr(_Base, '_p2_switch', False): _Base = _Base.__mro__[1]
MONITOR = {'v': 'val_loss'}
class SwitchPlateau(_Base):
    _p2_switch = True
    def __init__(self, monitor='val_loss', **kw):
        super().__init__(monitor=MONITOR['v'], **kw)
tf.keras.callbacks.ReduceLROnPlateau = SwitchPlateau
def use_schedule(s):
    MONITOR['v'] = 'loss' if s == 'train' else 'val_loss'
    assert qseg2.tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=4).monitor == MONITOR['v']
for s in ('train', 'val'):
    use_schedule(s); print(s, '-> ReduceLROnPlateau monitors', qseg2.tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=4).monitor)


In [ ]:
# 5 · Training (resumable; the two sessions share the work through lock files)
GP = ['g_q', 'g_cl', 'g_id', 'p_q', 'p_cl', 'p_id']
def run_list(kind):
    if kind == 'primary':
        runs = [(m, f, 0) for f in range(5) for m in ['unet', 'full'] + GP]
        runs += [(m, f, s) for s in (1, 2) for f in range(5) for m in GP]
    else:
        runs = [(m, f, s) for s in (0, 1, 2) for f in range(5) for m in GP]
    return runs
DATA = (X, Y, DF)
LOCK_H = 4
def claim(root, run):
    d = f'{root}/{run}'
    if os.path.exists(f'{d}/summary.json'): return False
    lk = f'{d}/.lock'
    if os.path.exists(lk):
        who, t = open(lk).read().split()
        if who != SESSION and time.time() - float(t) < LOCK_H * 3600: return False
    os.makedirs(d, exist_ok=True); open(lk, 'w').write(f'{SESSION} {time.time()}')
    time.sleep(20)                                 # Drive needs a moment to show the other session's lock
    return open(lk).read().split()[0] == SESSION
def lr_check(root, run):
    h = pd.read_csv(f'{root}/{run}/history.csv')
    col = 'learning_rate' if 'learning_rate' in h else 'lr'
    low = h.index[h[col] <= 1e-6]
    return (int(low[0]) + 1) if len(low) else None, len(h)
def train_set(kind, sched):
    root = ROOT[sched]; use_schedule(sched)
    runs = run_list(kind); cfg.to_json(f'{root}/train_config.json')
    json.dump({'schedule': sched, 'monitor': MONITOR['v'], 'set': kind}, open(f'{root}/schedule.json', 'w'))
    if SESSION == 'B': runs = runs[::-1]          # A works from the front, B from the back
    for name, fold, seed in runs:
        run = f'{name}_fold{fold}_seed{seed}'
        if not claim(root, run): continue
        qseg2.run_fold(cfg, DATA, name, fold, seed, out_root=root, min_epochs=15, verbose=0)
        shrink_weights(run, root)
        ep_low, n_ep = lr_check(root, run)
        done = sum(os.path.exists(f'{root}/{m}_fold{f}_seed{s}/summary.json') for m, f, s in run_list(kind))
        print(time.strftime('%H:%M UTC', time.gmtime()), f'| {kind}/{sched} {run}: {n_ep} epochs, lr<=1e-6 from epoch {ep_low} '
              f'| finished {done} / {len(runs)}', flush=True)
    left = [r for r in run_list(kind) if not os.path.exists(f'{root}/{r[0]}_fold{r[1]}_seed{r[2]}/summary.json')]
    print(f'{kind} set: nothing left for this session; {len(left)} run(s) still running in the other session', flush=True)
train_set('primary', PRIMARY)
if RUN_SENSITIVITY:
    train_set('sensitivity', OTHER)
print('all requested runs finished')


In [ ]:
# 6 · Quick table (mean per-image Dice over finished runs)
for s in ('train', 'val'):
    r = qseg2.collect(ROOT[s])
    if len(r):
        t = r.groupby('model')[['dice_per_image', 'dice_sever_const', 'dice_sever_swap', 'epochs_run']].mean().round(4)
        t['n_runs'] = r.groupby('model').size()
        print(f'schedule {s}:'); print(t.reindex([m for m in qseg2.MODELS2 if m in t.index]).to_string()); print()


In [ ]:
# 7 · Auto-shutdown: save Drive, then release the GPU (runs only after cells 5 and 6 have finished)
import time
from google.colab import drive, runtime
print('training cell finished at', time.strftime('%H:%M:%S UTC', time.gmtime()))
drive.flush_and_unmount()
print('Drive flushed')
time.sleep(30)
runtime.unassign()
